# Getting Started with Goqu

This notebook introduces Goqu's core features: building quantum circuits,
visualizing them as inline SVG diagrams, and running simulations.

**Requirements:** [gonb](https://github.com/janpfeifer/gonb) Go Jupyter kernel.

```
go install github.com/janpfeifer/gonb@latest && gonb --install
```

In [ ]:
import (
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/sim/statevector"
)

In [ ]:
var (
	bell *ir.Circuit
	ghz  *ir.Circuit
)

## Building a Bell State

A Bell state is the simplest entangled circuit: a Hadamard gate followed by a CNOT.

In [ ]:
%%
bell, _ = builder.New("bell", 2).H(0).CNOT(0, 1).Build()
fmt.Println(draw.String(bell))

## Inline SVG Visualization

`draw.SVG()` returns an SVG string that `gonbui.DisplaySvg()` renders inline.

In [ ]:
%%
gonbui.DisplaySvg(draw.SVG(bell))

### Dark Theme

Pass `draw.WithStyle(draw.DarkStyle())` for a dark background.

In [ ]:
%%
gonbui.DisplaySvg(draw.SVG(bell, draw.WithStyle(draw.DarkStyle())))

## Statevector Simulation

Use `statevector.New()` to create a simulator, then `Evolve()` to apply the
circuit without measurement collapse.

In [ ]:
%%
sim := statevector.New(2)
sim.Evolve(bell)
for i, amp := range sim.StateVector() {
	if p := real(amp)*real(amp) + imag(amp)*imag(amp); p > 1e-10 {
		fmt.Printf("|%02b> amplitude: %.4f  probability: %.1f\n", i, amp, math.Round(p*10)/10)
	}
}

## Shot-Based Sampling

Add measurements and sample 1000 shots. The circuit needs `MeasureAll()` for `Run()`.

In [ ]:
%%
bellM, _ := builder.New("bell-measured", 2).H(0).CNOT(0, 1).MeasureAll().Build()
sim2 := statevector.New(2)
counts, _ := sim2.Run(bellM, 1000)
for state, n := range counts {
	fmt.Printf("|%s>: %d\n", state, n)
}

## GHZ State

A GHZ state generalizes entanglement to 3+ qubits: H on qubit 0, then CNOT to each subsequent qubit.

In [ ]:
%%
ghz, _ = builder.New("ghz", 3).H(0).CNOT(0, 1).CNOT(0, 2).Build()
gonbui.DisplaySvg(draw.SVG(ghz))

In [ ]:
%%
simGHZ := statevector.New(3)
simGHZ.Evolve(ghz)
for i, amp := range simGHZ.StateVector() {
	if p := real(amp)*real(amp) + imag(amp)*imag(amp); p > 1e-10 {
		fmt.Printf("|%03b> probability: %.2f\n", i, p)
	}
}

## Circuit Statistics

`Stats()` returns depth, gate count, two-qubit gates, and more.

In [ ]:
%%
s := ghz.Stats()
fmt.Printf("Depth:          %d\n", s.Depth)
fmt.Printf("Gate count:     %d\n", s.GateCount)
fmt.Printf("Two-qubit gates: %d\n", s.TwoQubitGates)